# AIC — Notebook 03: Caption Generation (Qwen2.5-VL)

Generates bilingual EN+VI captions for all keyframes.

**GPU required:** T4 (16GB) with `load_in_4bit=True`  
**Estimated time:** ~6-12 hours for full dataset

**Output:** `/kaggle/working/captions/L{XX}_{V}.json`

In [ ]:
import subprocess, sys, os
GITHUB_REPO = "https://github.com/YOUR_USERNAME/AIC_System.git"
REPO_DIR = "/kaggle/working/AIC_System"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git","clone","--depth","1",GITHUB_REPO,REPO_DIR],check=True)
else:
    subprocess.run(["git","-C",REPO_DIR,"pull"],check=True)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",f"{REPO_DIR}/requirements.txt"],check=True)
print('Setup complete.')

In [ ]:
from pathlib import Path
DATASET_SLUG = "your-username/aic-hcmc-data"  # ← change
DATASET_NAME = DATASET_SLUG.split('/')[-1]
DATASET_PATH = Path(f"/kaggle/input/{DATASET_NAME}")
MAP_KF_DIR   = DATASET_PATH / "map-keyframes-aic25-b1" / "map-keyframes"
KF_IMG_ROOT  = DATASET_PATH / "keyframes" / "keyframes"
OUTPUT_DIR   = Path("/kaggle/working/captions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
csv_files = sorted(MAP_KF_DIR.glob("*.csv"))
print(f'{len(csv_files)} videos to caption')

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
# Use 4-bit if VRAM < 15GB
LOAD_4BIT = torch.cuda.get_device_properties(0).total_memory < 15 * 1024**3
print(f'4-bit quantization: {LOAD_4BIT}')

In [ ]:
from src.feature_extractors.captioner import CaptionGenerator
captioner = CaptionGenerator(
    model_name='Qwen/Qwen2.5-VL-7B-Instruct',
    device='cuda',
    load_in_4bit=LOAD_4BIT,
    max_new_tokens=150,
    generate_vi=True,
)
captioner.load()
print('Qwen2.5-VL ready.')

In [ ]:
from tqdm import tqdm
errors = []
for csv_path in tqdm(csv_files, desc='Caption Generation'):
    video_id = csv_path.stem
    try:
        captioner.extract_video(
            video_id=video_id,
            keyframes_dir=str(KF_IMG_ROOT),
            map_keyframes_csv=str(csv_path),
            output_dir=str(OUTPUT_DIR),
            overwrite=False,
        )
    except Exception as e:
        errors.append((video_id, str(e)))
        print(f'ERROR: {video_id}: {e}')
print(f'Done. Errors: {len(errors)}')
print(f'Output files: {len(list(OUTPUT_DIR.glob("*.json")))}')